# GP and spark

In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm
pd.set_option('Display.max_columns', None)
import re
import sys
from tsu_spark_utils import *

In [2]:
import os
def get_spark_session(name, level):
    """
    Get spark context
    :: name - set your app name
    :: level - set max resources level
    """
    python_path = sys.executable
    kernel = python_path.split('/')[-3]
    os.environ['SPARK_MAJOR_VERSION'] = '3'
    os.environ['SPARK_HOME'] = '/usr/sdp/current/spark3-client/'
    os.environ['PYSPARK_DRIVER_PYTHON'] = python_path
    os.environ['PYSPARK_PYTHON'] = python_path
    os.environ['LD_LIBRARY_PATH'] = '/opt/python/virtualenv/jupyter/lib'
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/')
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/lib/py4j_current')
 
    # Resources Level Profiles                           #  cpu --  ram -- desc
    if level == 1: lv = ['basic',2,10,2,10,2,2,10]       #   21 --  142 -- для базовых запросов (show create table tbl, show partitions tbl)
    if level == 2: lv = ['basic+CPU',2,10,2,10,2,2,20]   #   41 --  262 -- для простой аналитики (select * from limit 100, sum/count/avg)
    if level == 3: lv = ['middle',4,28,6,28,6,4,20]      #   81 --  742 -- для агрегатов за период 1-2мес (client_aggr_mnth, epk_campaign_daily)
    if level == 4: lv = ['middle+CPU',4,28,6,28,6,4,25]  #  101 --  912 -- для агрегатов за период >1-6мес  (client_aggr_mnth, epk_campaign_daily)
    if level == 5: lv = ['high',4,28,6,36,8,6,30]        #  121 -- 1100 -- для детальных таблиц с большими партициями (_sbol, _card, _eps)
    if level == 6: lv = ['high+CPU',4,18,5,36,8,6,40]    #  161 -- 1000 -- для детальных таблиц с мелкими партициями (feedbacks)
    if level == 7: lv = ['unfriendly',5,28,6,44,10,8,40] #  201 -- 1458 -- для запуска вечером/ночью или на пустом кластере (не рекомендуется)
    lvname = f'{level}.{lv[0]}({lv[1]*lv[7]+1},{lv[4]+lv[2]*lv[7]})'
    print(f'Kernel: {kernel}, Python_path: {python_path}, Resource_level: {lvname}')
    
    # Spark Config      
    from pyspark import SparkContext, SparkConf
    from pyspark.sql import SparkSession
  
    conf = SparkConf().setAppName(f'{name} \n ::{kernel}::{lvname}::')\
        .setMaster("yarn")\
        .set('spark.executor.cores',                     f'{lv[1]}')\
        .set('spark.executor.memory',                    f'{lv[2]}g')\
        .set('spark.executor.memoryOverhead',            f'{lv[3]}g')\
        .set('spark.driver.memory',                      f'{lv[4]}g')\
        .set('spark.driver.memoryOverhead',              f'{lv[5]}g')\
        .set('spark.driver.maxResultSize', '10g')\
        .set('spark.dynamicAllocation.initialExecutors', f'{lv[6]}')\
        .set('spark.dynamicAllocation.maxExecutors',     f'{lv[7]}')\
        .set('spark.dynamicAllocation.enabled', 'true')\
        .set('spark.dynamicAllocation.executorIdleTimeout', '120s')\
        .set('spark.dynamicAllocation.cachedExecutorIdleTimeout', '600s')\
        .set('spark.hive.mapred.supports.subdirectories', 'true')\
        .set('spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive', 'true')\
        .set('spark.shuffle.service.enabled', 'true')\
        .set('spark.port.maxRetries', '150')\
       .set('spark.sql.parquet.writeLegacyFormat', 'true')\
        .set('spark.kerberos.access.hadoopFileSystems','hdfs://arnsdpsbx:8020/')\
        .set('spark.sql.autoBroadcastJoinThreshold','20971520')
    
    spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()
    return spark

try: spark
except NameError: print('Spark3 Starting')
else:
    print('Spark3 Restarting')
    spark.stop()
    
spark = get_spark_session('platon_mvs_analytics', 5) # For example, MyPySpark3
  
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pyspark.sql.types as T
  
sc = spark.sparkContext
sc.setLogLevel('OFF')  # or 'INFO' or 'WARN' or 'OFF'
spark


Spark3 Starting
Kernel: mlpy3811v23, Python_path: /data/sdp/mlpy3811v23/bin/python, Resource_level: 5.high(121,876)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/17 19:56:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/17 19:56:18 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/17 19:56:18 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/03/17 19:56:18 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/03/17 19:56:22 WARN HiveConf: HiveConf of name hive.mapred.supports.subdirectories does not exist
26/03/17 19:56:22 WARN Client: Exception encountered while connecting to the server 
org.apache.hadoop.ipc.RemoteException(org.apache.hadoop.ipc.StandbyException): Operation category READ is not supported in state standby. Visit https://s.apache.org/sbnn-error
	at org.apache.hadoop.security.SaslRpcClient.saslConnect(SaslRpcClient.java:376)
	at org.apache.hadoop.ipc.Clien

In [3]:
gp_kinit()
dbh, conn, cur = get_gp_connect()

kinit: Password incorrect while getting initial credentials


('21417984',) connection done


# Analytics

## Telemarketing

In [4]:
df_october_september = pd.read_sql('''
with first as (select epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-09-30' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select clientid, predlozhenie, ppp_30, ppd_30_day_sale, splits, producttype, campain, x_btn_connect_date 
from prx_bpm_telemarketing_analitika_s_grnplm_as_rozn_anofl_view_3t2."v_anofl_rep$_coldfun_sale_deteil_dp1_21417984_view" a 
inner join first b on a.clientid = b.epk_id
where x_btn_connect_date >= '2024-10-01' and x_btn_connect_date <= '2025-09-30'
and splits not in ('SC_SaleKr','SaleKR_TM','Hot_sales', 'Sale_TM')   -- условие которым исключаем сплиты горячего потока
and coalesce (producttype,'') <> 'Изменение лимита'  --исключаем кампанию, тк кампания сервисная
and coalesce (producttype,'') <> 'Мегамаркет'   --исключаем кампанию, тк кампания сервисная
and campain is not null   -- учитываются только клиенты с разметкой в кампании, расчет всех метрик  учетом этого условия
;
''', conn)

/tmp/ipykernel_94048/1857059856.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_october_september = pd.read_sql('''


In [5]:
df_october_september = df_october_september.drop_duplicates()

In [6]:
df_october_september.to_parquet('telemarketing_october_september.parquet')

In [9]:
df_august_july = pd.read_sql('''
with first as (select epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-07-31' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select clientid, predlozhenie, ppp_30, ppd_30_day_sale, splits, producttype, campain, x_btn_connect_date 
from prx_bpm_telemarketing_analitika_s_grnplm_as_rozn_anofl_view_3t2."v_anofl_rep$_coldfun_sale_deteil_dp1_21417984_view" a 
inner join first b on a.clientid = b.epk_id
where x_btn_connect_date >= '2024-08-01' and x_btn_connect_date <= '2025-07-31'
and splits not in ('SC_SaleKr','SaleKR_TM','Hot_sales', 'Sale_TM')   -- условие которым исключаем сплиты горячего потока
and coalesce (producttype,'') <> 'Изменение лимита'  --исключаем кампанию, тк кампания сервисная
and coalesce (producttype,'') <> 'Мегамаркет'   --исключаем кампанию, тк кампания сервисная
and campain is not null   -- учитываются только клиенты с разметкой в кампании, расчет всех метрик  учетом этого условия
;
''', conn)

/tmp/ipykernel_94048/3404459030.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_august_july = pd.read_sql('''


In [10]:
df_august_july = df_august_july.drop_duplicates()

In [11]:
df_august_july.to_parquet('telemarketing_august_july.parquet')

In [13]:
df_june_may = pd.read_sql('''
with first as (select epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-05-31' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select clientid, predlozhenie, ppp_30, ppd_30_day_sale, splits, producttype, campain, x_btn_connect_date 
from prx_bpm_telemarketing_analitika_s_grnplm_as_rozn_anofl_view_3t2."v_anofl_rep$_coldfun_sale_deteil_dp1_21417984_view" a 
inner join first b on a.clientid = b.epk_id
where x_btn_connect_date >= '2024-06-01' and x_btn_connect_date <= '2025-05-31'
and splits not in ('SC_SaleKr','SaleKR_TM','Hot_sales', 'Sale_TM')   -- условие которым исключаем сплиты горячего потока
and coalesce (producttype,'') <> 'Изменение лимита'  --исключаем кампанию, тк кампания сервисная
and coalesce (producttype,'') <> 'Мегамаркет'   --исключаем кампанию, тк кампания сервисная
and campain is not null   -- учитываются только клиенты с разметкой в кампании, расчет всех метрик  учетом этого условия
;
''', conn)

/tmp/ipykernel_94048/2893592710.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_june_may = pd.read_sql('''


In [14]:
df_june_may = df_june_may.drop_duplicates()

In [15]:
df_june_may.to_parquet('telemarketing_june_may.parquet')

## Звонки

In [8]:
q = f'''
DROP TABLE IF EXISTS s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_october_september;
    
CREATE TABLE s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_october_september as (
WITH base as (
    select epk_id, main_call, channel, presentation_flag, contact_flag, sale_dt_1prod, sale_dt_2prod
from prx_bpm_zvonki_i_prodazhi_gp_s_grnplm_as_rozn_anofl_view_20t."v_anofl_rep$_tm_funnel_dp1_21417984_view"
where main_call between '2024-10-01' and '2025-09-30'
)
SELECT
*
FROM base
)
'''
conn.rollback()
cur.execute(q)
conn.commit()

In [9]:
q = f'''
DROP TABLE IF EXISTS s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_august_july;
    
CREATE TABLE s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_august_july as (
WITH base as (
    select epk_id, main_call, channel, presentation_flag, contact_flag, sale_dt_1prod, sale_dt_2prod
from prx_bpm_zvonki_i_prodazhi_gp_s_grnplm_as_rozn_anofl_view_20t."v_anofl_rep$_tm_funnel_dp1_21417984_view"
where main_call between '2024-08-01' and '2025-07-31'
)
SELECT
*
FROM base
)
'''
conn.rollback()
cur.execute(q)
conn.commit()

In [10]:
q = f'''
DROP TABLE IF EXISTS s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_june_may;
    
CREATE TABLE s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_june_may as (
WITH base as (
    select epk_id, main_call, channel, presentation_flag, contact_flag, sale_dt_1prod, sale_dt_2prod
from prx_bpm_zvonki_i_prodazhi_gp_s_grnplm_as_rozn_anofl_view_20t."v_anofl_rep$_tm_funnel_dp1_21417984_view"
where main_call between '2024-06-01' and '2025-05-31'
)
SELECT
*
FROM base
)
'''
conn.rollback()
cur.execute(q)
conn.commit()

In [11]:
to_pxf = f'''
CREATE WRITABLE EXTERNAL TABLE s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_october_september_ext
(
   LIKE s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_october_september
)
LOCATION ('pxf://user/team/team_ss/calls_mvs_october_september?PROFILE=hdfs:parquet&SERVER=sbx_sdp_ld_rozn_electron')
ON ALL
FORMAT 'custom' (formatter = 'pxfwritable_export')
ENCODING = 6
DISTRIBUTED randomly;

CREATE WRITABLE EXTERNAL TABLE s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_august_july_ext
(
   LIKE s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_august_july 
)
LOCATION ('pxf://user/team/team_ss/calls_mvs_august_july?PROFILE=hdfs:parquet&SERVER=sbx_sdp_ld_rozn_electron')
ON ALL
FORMAT 'custom' (formatter = 'pxfwritable_export')
ENCODING = 6
DISTRIBUTED randomly;

CREATE WRITABLE EXTERNAL TABLE s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_june_may_ext
(
   LIKE s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_june_may 
)
LOCATION ('pxf://user/team/team_ss/calls_mvs_june_may?PROFILE=hdfs:parquet&SERVER=sbx_sdp_ld_rozn_electron')
ON ALL
FORMAT 'custom' (formatter = 'pxfwritable_export')
ENCODING = 6
DISTRIBUTED randomly;

INSERT INTO s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_october_september_ext
SELECT * FROM s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_october_september;

INSERT INTO s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_august_july_ext
SELECT * FROM s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_august_july;

INSERT INTO s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_june_may_ext
SELECT * FROM s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_june_may;

drop EXTERNAL table s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_october_september_ext;
drop EXTERNAL table s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_august_july_ext;
drop EXTERNAL table s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_june_may_ext;
'''
cur.execute(to_pxf)
conn.commit()

## KM

In [3]:
df_october_september = pd.read_sql('''
select 
epk_id, activity_dt,
sum(case when activity_type = 'Исходящий звонок' and activity_status_detail in ('Звонок состоялся', 'Звонок отработан') then 1 else 0 end) as is_call_km,  -- звонков
sum(case when activity_type = 'Исходящий звонок' and activity_status_detail in ('Звонок состоялся', 'Звонок отработан') and date_part('epoch', factend_dttm - factstart_dttm) >= 60 then 1 else 0 end) as is_call_s_km,  --дозвоны
sum(case when activity_status_detail = 'Встреча состоялась' and date_part('epoch', factend_dttm - factstart_dttm) >= 60 then 1 else 0 end) as is_meet_km --встречи
from s_grnplm_ld_rozn_electron_mvs.prmr_data_pprb_activity a_p
where 1=1 
and activity_dt between '2024-10-01' and '2025-09-30'
and a_p.role_from_sap = 'ПРЕМЬЕР' and a_p.activity_status in ('Выполнена', 'Завершена', 'Закрыта')
and a_p.activity_type in ('Исходящий звонок','Внутренняя встреча')
group by 1,2

''', conn)

/tmp/ipykernel_113533/254085752.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_october_september = pd.read_sql('''


In [4]:
df_october_september = df_october_september.drop_duplicates()

In [5]:
df_october_september.to_parquet('km_october_september.parquet')

In [7]:
df_august_july = pd.read_sql('''
select 
epk_id, activity_dt,
sum(case when activity_type = 'Исходящий звонок' and activity_status_detail in ('Звонок состоялся', 'Звонок отработан') then 1 else 0 end) as is_call_km,  -- звонков
sum(case when activity_type = 'Исходящий звонок' and activity_status_detail in ('Звонок состоялся', 'Звонок отработан') and date_part('epoch', factend_dttm - factstart_dttm) >= 60 then 1 else 0 end) as is_call_s_km,  --дозвоны
sum(case when activity_status_detail = 'Встреча состоялась' and date_part('epoch', factend_dttm - factstart_dttm) >= 60 then 1 else 0 end) as is_meet_km --встречи
from s_grnplm_ld_rozn_electron_mvs.prmr_data_pprb_activity a_p
where 1=1 
and activity_dt between '2024-08-01' and '2025-07-31'
and a_p.role_from_sap = 'ПРЕМЬЕР' and a_p.activity_status in ('Выполнена', 'Завершена', 'Закрыта')
and a_p.activity_type in ('Исходящий звонок','Внутренняя встреча')
group by 1,2

''', conn)

/tmp/ipykernel_113533/3562424488.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_august_july = pd.read_sql('''


In [8]:
df_august_july = df_august_july.drop_duplicates()

In [9]:
df_august_july.to_parquet('km_august_july.parquet')

In [10]:
df_june_may = pd.read_sql('''
select 
epk_id, activity_dt,
sum(case when activity_type = 'Исходящий звонок' and activity_status_detail in ('Звонок состоялся', 'Звонок отработан') then 1 else 0 end) as is_call_km,  -- звонков
sum(case when activity_type = 'Исходящий звонок' and activity_status_detail in ('Звонок состоялся', 'Звонок отработан') and date_part('epoch', factend_dttm - factstart_dttm) >= 60 then 1 else 0 end) as is_call_s_km,  --дозвоны
sum(case when activity_status_detail = 'Встреча состоялась' and date_part('epoch', factend_dttm - factstart_dttm) >= 60 then 1 else 0 end) as is_meet_km --встречи
from s_grnplm_ld_rozn_electron_mvs.prmr_data_pprb_activity a_p
where 1=1 
and activity_dt between '2024-06-01' and '2025-05-31'
and a_p.role_from_sap = 'ПРЕМЬЕР' and a_p.activity_status in ('Выполнена', 'Завершена', 'Закрыта')
and a_p.activity_type in ('Исходящий звонок','Внутренняя встреча')
group by 1,2

''', conn)

/tmp/ipykernel_113533/3906388505.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_june_may = pd.read_sql('''


In [11]:
df_june_may = df_june_may.drop_duplicates()

In [12]:
df_june_may.to_parquet('km_june_may.parquet')

## ДКМ

In [13]:
df_october_september = pd.read_sql('''
with first as (select distinct epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-09-30' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select a.clientid as epk_id, --формат text
"Дата" as activity_dt,
sum(case when a."время звонка" > 0 then 1 else 0 end) as is_call_dkm,
sum(case when a."время звонка" >= 1 or (a."время звонка" > 0 and a."Статус звонка" = 'Дозвон') then 1 else 0 end) as is_call_s_dkm,
0 as is_meet_dkm  --нет встреч
from s_grnplm_ld_rozn_electron_mvs.v_dkm_calls a
where a."Дата" between '2024-10-01' and '2025-09-30'
group by 1,2
''', conn)

/tmp/ipykernel_113533/2996268822.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_october_september = pd.read_sql('''


In [14]:
df_october_september = df_october_september.drop_duplicates()

In [15]:
df_october_september.to_parquet('dkm_october_september.parquet')

In [16]:
df_august_july = pd.read_sql('''
with first as (select distinct epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-07-31' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select a.clientid as epk_id, --формат text
"Дата" as activity_dt,
sum(case when a."время звонка" > 0 then 1 else 0 end) as is_call_dkm,
sum(case when a."время звонка" >= 1 or (a."время звонка" > 0 and a."Статус звонка" = 'Дозвон') then 1 else 0 end) as is_call_s_dkm,
0 as is_meet_dkm  --нет встреч
from s_grnplm_ld_rozn_electron_mvs.v_dkm_calls a
where a."Дата" between '2024-08-01' and '2025-07-31'
group by 1,2
''', conn)

/tmp/ipykernel_113533/1047973081.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_august_july = pd.read_sql('''


In [17]:
df_august_july = df_august_july.drop_duplicates()

In [18]:
df_august_july.to_parquet('dkm_august_july.parquet')

In [19]:
df_june_may = pd.read_sql('''
with first as (select distinct epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-05-31' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select a.clientid as epk_id, --формат text
"Дата" as activity_dt,
sum(case when a."время звонка" > 0 then 1 else 0 end) as is_call_dkm,
sum(case when a."время звонка" >= 1 or (a."время звонка" > 0 and a."Статус звонка" = 'Дозвон') then 1 else 0 end) as is_call_s_dkm,
0 as is_meet_dkm  --нет встреч
from s_grnplm_ld_rozn_electron_mvs.v_dkm_calls a
where a."Дата" between '2024-06-01' and '2025-05-31'
group by 1,2
''', conn)

/tmp/ipykernel_113533/3380017257.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_june_may = pd.read_sql('''


In [20]:
df_june_may = df_june_may.drop_duplicates()

In [21]:
df_june_may.to_parquet('dkm_june_may.parquet')

## Sales MVS СБОЛ и ВСП

### SBOL

In [12]:
q = f'''
DROP TABLE IF EXISTS s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_october_september;
    
CREATE TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_october_september as (
with first as (select distinct epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-09-30' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select epk_id, sale_dt, product_class_name, sale_channel_name from s_grnplm_vd_rozn_mpp_aaas_vd.ft_prod_sales
inner join first using(epk_id)
where sale_dt between '2024-10-01' and '2025-09-30'
and product_class_name in (
    'Запрос кредитной истории',
    'СберСпасибо',
    'Текущий счет',
    'Зарплатный клиент',
    'Монеты и слитки',
    'Страховой продукт',
    'Брокерское обслуживание (CIB)',
    'Платежи',
    'Срочный депозит',
    'Подписка',
    'Металлический счет',
    'Маркировка в канале Премьер/VIP',
    'Карточный счет',
    'Потребительский кредит',
    'Пакет услуг Премьер',
    'ПИФ/ДУ',
    'АП ЖКХ (новая система)',
    'Переводы',
    'Открытие счета ЮЛ',
    'Подключение транзакционных пушей',
    'Облигация',
    'Кредитная карта',
    'Автокредит',
    'Пенсионные начисления'
    )
and sale_channel_name in ( 
    'Интернет-банк (СБОЛ)',
    'СБОЛ.Про',
    'Сайт',
    'СБОЛ',
    'Интернет сайт',
    'MOBILE СБОЛ',
    'SBOL PRO',
    'SBOL Mob',
    'SBOL Web',
    'САЙТ',
    'СБОЛ МП',
    'СБЕРБАНК ПРЕМЬЕР-SBOL PRO',
    'Мобильное приложение СБОЛ',
    'СБОЛ САЙТ',
    'WEB СБОЛ',
    'Сайт Банка(Гостевой СБОЛ)',
    'Мобильный банк'
    )
)
'''
conn.rollback()
cur.execute(q)
conn.commit()

In [13]:
q = f'''
DROP TABLE IF EXISTS s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_august_july;
    
CREATE TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_august_july as (
with first as (select distinct epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-07-31' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select epk_id, sale_dt, product_class_name, sale_channel_name from s_grnplm_vd_rozn_mpp_aaas_vd.ft_prod_sales
inner join first using(epk_id)
where sale_dt between '2024-08-01' and '2025-07-31'
and product_class_name in (
    'Запрос кредитной истории',
    'СберСпасибо',
    'Текущий счет',
    'Зарплатный клиент',
    'Монеты и слитки',
    'Страховой продукт',
    'Брокерское обслуживание (CIB)',
    'Платежи',
    'Срочный депозит',
    'Подписка',
    'Металлический счет',
    'Маркировка в канале Премьер/VIP',
    'Карточный счет',
    'Потребительский кредит',
    'Пакет услуг Премьер',
    'ПИФ/ДУ',
    'АП ЖКХ (новая система)',
    'Переводы',
    'Открытие счета ЮЛ',
    'Подключение транзакционных пушей',
    'Облигация',
    'Кредитная карта',
    'Автокредит',
    'Пенсионные начисления'
    )
and sale_channel_name in ( 
    'Интернет-банк (СБОЛ)',
    'СБОЛ.Про',
    'Сайт',
    'СБОЛ',
    'Интернет сайт',
    'MOBILE СБОЛ',
    'SBOL PRO',
    'SBOL Mob',
    'SBOL Web',
    'САЙТ',
    'СБОЛ МП',
    'СБЕРБАНК ПРЕМЬЕР-SBOL PRO',
    'Мобильное приложение СБОЛ',
    'СБОЛ САЙТ',
    'WEB СБОЛ',
    'Сайт Банка(Гостевой СБОЛ)',
    'Мобильный банк'
    )
)
'''
conn.rollback()
cur.execute(q)
conn.commit()

In [14]:
q = f'''
DROP TABLE IF EXISTS s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_june_may;
    
CREATE TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_june_may as (
with first as (select distinct epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-05-31' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select epk_id, sale_dt, product_class_name, sale_channel_name from s_grnplm_vd_rozn_mpp_aaas_vd.ft_prod_sales
inner join first using(epk_id)
where sale_dt between '2024-06-01' and '2025-05-31'
and product_class_name in (
    'Запрос кредитной истории',
    'СберСпасибо',
    'Текущий счет',
    'Зарплатный клиент',
    'Монеты и слитки',
    'Страховой продукт',
    'Брокерское обслуживание (CIB)',
    'Платежи',
    'Срочный депозит',
    'Подписка',
    'Металлический счет',
    'Маркировка в канале Премьер/VIP',
    'Карточный счет',
    'Потребительский кредит',
    'Пакет услуг Премьер',
    'ПИФ/ДУ',
    'АП ЖКХ (новая система)',
    'Переводы',
    'Открытие счета ЮЛ',
    'Подключение транзакционных пушей',
    'Облигация',
    'Кредитная карта',
    'Автокредит',
    'Пенсионные начисления'
    )
and sale_channel_name in ( 
    'Интернет-банк (СБОЛ)',
    'СБОЛ.Про',
    'Сайт',
    'СБОЛ',
    'Интернет сайт',
    'MOBILE СБОЛ',
    'SBOL PRO',
    'SBOL Mob',
    'SBOL Web',
    'САЙТ',
    'СБОЛ МП',
    'СБЕРБАНК ПРЕМЬЕР-SBOL PRO',
    'Мобильное приложение СБОЛ',
    'СБОЛ САЙТ',
    'WEB СБОЛ',
    'Сайт Банка(Гостевой СБОЛ)',
    'Мобильный банк'
    )
)
'''
conn.rollback()
cur.execute(q)
conn.commit()

In [15]:
to_pxf = f'''


CREATE WRITABLE EXTERNAL TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_october_september_ext
(
   LIKE s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_october_september
)
LOCATION ('pxf://user/team/team_ss/sales_sbol_mvs_october_september?PROFILE=hdfs:parquet&SERVER=sbx_sdp_ld_rozn_electron')
ON ALL
FORMAT 'custom' (formatter = 'pxfwritable_export')
ENCODING = 6
DISTRIBUTED randomly;

CREATE WRITABLE EXTERNAL TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_august_july_ext
(
   LIKE s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_august_july 
)
LOCATION ('pxf://user/team/team_ss/sales_sbol_mvs_august_july?PROFILE=hdfs:parquet&SERVER=sbx_sdp_ld_rozn_electron')
ON ALL
FORMAT 'custom' (formatter = 'pxfwritable_export')
ENCODING = 6
DISTRIBUTED randomly;

CREATE WRITABLE EXTERNAL TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_june_may_ext
(
   LIKE s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_june_may 
)
LOCATION ('pxf://user/team/team_ss/sales_sbol_mvs_june_may?PROFILE=hdfs:parquet&SERVER=sbx_sdp_ld_rozn_electron')
ON ALL
FORMAT 'custom' (formatter = 'pxfwritable_export')
ENCODING = 6
DISTRIBUTED randomly;

INSERT INTO s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_october_september_ext
SELECT * FROM s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_october_september;

INSERT INTO s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_august_july_ext
SELECT * FROM s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_august_july;

INSERT INTO s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_june_may_ext
SELECT * FROM s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_june_may;

drop EXTERNAL table s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_october_september_ext;
drop EXTERNAL table s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_august_july_ext;
drop EXTERNAL table s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_june_may_ext;
'''
cur.execute(to_pxf)
conn.commit()

### VSP

In [16]:
q = f'''
DROP TABLE IF EXISTS s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_october_september;
    
CREATE TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_october_september as (
with first as (select distinct epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-09-30' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select epk_id, sale_dt, product_class_name, sale_channel_name from s_grnplm_vd_rozn_mpp_aaas_vd.ft_prod_sales
inner join first using(epk_id)
where sale_dt between '2024-10-01' and '2025-09-30'
and product_class_name in (
    'Запрос кредитной истории',
    'СберСпасибо',
    'Текущий счет',
    'Зарплатный клиент',
    'Монеты и слитки',
    'Страховой продукт',
    'Брокерское обслуживание (CIB)',
    'Платежи',
    'Срочный депозит',
    'Подписка',
    'Металлический счет',
    'Маркировка в канале Премьер/VIP',
    'Карточный счет',
    'Потребительский кредит',
    'Пакет услуг Премьер',
    'ПИФ/ДУ',
    'АП ЖКХ (новая система)',
    'Переводы',
    'Открытие счета ЮЛ',
    'Подключение транзакционных пушей',
    'Облигация',
    'Кредитная карта',
    'Автокредит',
    'Пенсионные начисления'
    )
and sale_channel_name in ( 
    'ПРЯМЫЕ ПРОДАЖИ',
    'СБ1',
    'Масс',
    'ЕФС ВИП обслуживание',
    'Прямые продажи на предприятии (DSA)',
    'Сайт',
    'SB1',
    'PRIVATEBANKING',
    'ПРЯМЫЕ ПРОДАЖИ ВИП',
    'Premier',
    'САЙТ',
    'Клиентская зона',
    'PB',
    'MASS',
    'ВСП',
    'VIP',
    'ВСП Премьер',
    'ВИП ВСП',
    'Премьер',
    'ВСП. Мотивация',
    'Офисы Private Banking',
    'СБЕРБАНК ПРЕМЬЕР-ПАО'
    )
order by random()
limit 30000000
)
'''
conn.rollback()
cur.execute(q)
conn.commit()

In [17]:
q = f'''
DROP TABLE IF EXISTS s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_august_july;
    
CREATE TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_august_july as (
with first as (select distinct epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-07-31' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select epk_id, sale_dt, product_class_name, sale_channel_name from s_grnplm_vd_rozn_mpp_aaas_vd.ft_prod_sales
inner join first using(epk_id)
where sale_dt between '2024-08-01' and '2025-07-31'
and product_class_name in (
    'Запрос кредитной истории',
    'СберСпасибо',
    'Текущий счет',
    'Зарплатный клиент',
    'Монеты и слитки',
    'Страховой продукт',
    'Брокерское обслуживание (CIB)',
    'Платежи',
    'Срочный депозит',
    'Подписка',
    'Металлический счет',
    'Маркировка в канале Премьер/VIP',
    'Карточный счет',
    'Потребительский кредит',
    'Пакет услуг Премьер',
    'ПИФ/ДУ',
    'АП ЖКХ (новая система)',
    'Переводы',
    'Открытие счета ЮЛ',
    'Подключение транзакционных пушей',
    'Облигация',
    'Кредитная карта',
    'Автокредит',
    'Пенсионные начисления'
    )
and sale_channel_name in ( 
    'ПРЯМЫЕ ПРОДАЖИ',
    'СБ1',
    'Масс',
    'ЕФС ВИП обслуживание',
    'Прямые продажи на предприятии (DSA)',
    'Сайт',
    'SB1',
    'PRIVATEBANKING',
    'ПРЯМЫЕ ПРОДАЖИ ВИП',
    'Premier',
    'САЙТ',
    'Клиентская зона',
    'PB',
    'MASS',
    'ВСП',
    'VIP',
    'ВСП Премьер',
    'ВИП ВСП',
    'Премьер',
    'ВСП. Мотивация',
    'Офисы Private Banking',
    'СБЕРБАНК ПРЕМЬЕР-ПАО'
    )
order by random()
limit 30000000
)
'''
conn.rollback()
cur.execute(q)
conn.commit()

In [18]:
q = f'''
DROP TABLE IF EXISTS s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_june_may;
    
CREATE TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_june_may as (
with first as (select distinct epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-05-31' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select epk_id, sale_dt, product_class_name, sale_channel_name from s_grnplm_vd_rozn_mpp_aaas_vd.ft_prod_sales
inner join first using(epk_id)
where sale_dt between '2024-06-01' and '2025-05-31'
and product_class_name in (
    'Запрос кредитной истории',
    'СберСпасибо',
    'Текущий счет',
    'Зарплатный клиент',
    'Монеты и слитки',
    'Страховой продукт',
    'Брокерское обслуживание (CIB)',
    'Платежи',
    'Срочный депозит',
    'Подписка',
    'Металлический счет',
    'Маркировка в канале Премьер/VIP',
    'Карточный счет',
    'Потребительский кредит',
    'Пакет услуг Премьер',
    'ПИФ/ДУ',
    'АП ЖКХ (новая система)',
    'Переводы',
    'Открытие счета ЮЛ',
    'Подключение транзакционных пушей',
    'Облигация',
    'Кредитная карта',
    'Автокредит',
    'Пенсионные начисления'
    )
and sale_channel_name in ( 
    'ПРЯМЫЕ ПРОДАЖИ',
    'СБ1',
    'Масс',
    'ЕФС ВИП обслуживание',
    'Прямые продажи на предприятии (DSA)',
    'Сайт',
    'SB1',
    'PRIVATEBANKING',
    'ПРЯМЫЕ ПРОДАЖИ ВИП',
    'Premier',
    'САЙТ',
    'Клиентская зона',
    'PB',
    'MASS',
    'ВСП',
    'VIP',
    'ВСП Премьер',
    'ВИП ВСП',
    'Премьер',
    'ВСП. Мотивация',
    'Офисы Private Banking',
    'СБЕРБАНК ПРЕМЬЕР-ПАО'
    )
order by random()
limit 30000000
)
'''
conn.rollback()
cur.execute(q)
conn.commit()

In [19]:
to_pxf = f'''


CREATE WRITABLE EXTERNAL TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_october_september_ext
(
   LIKE s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_october_september
)
LOCATION ('pxf://user/team/team_ss/sales_vsp_mvs_october_september?PROFILE=hdfs:parquet&SERVER=sbx_sdp_ld_rozn_electron')
ON ALL
FORMAT 'custom' (formatter = 'pxfwritable_export')
ENCODING = 6
DISTRIBUTED randomly;

CREATE WRITABLE EXTERNAL TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_august_july_ext
(
   LIKE s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_august_july 
)
LOCATION ('pxf://user/team/team_ss/sales_vsp_mvs_august_july?PROFILE=hdfs:parquet&SERVER=sbx_sdp_ld_rozn_electron')
ON ALL
FORMAT 'custom' (formatter = 'pxfwritable_export')
ENCODING = 6
DISTRIBUTED randomly;

CREATE WRITABLE EXTERNAL TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_june_may_ext
(
   LIKE s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_june_may 
)
LOCATION ('pxf://user/team/team_ss/sales_vsp_mvs_june_may?PROFILE=hdfs:parquet&SERVER=sbx_sdp_ld_rozn_electron')
ON ALL
FORMAT 'custom' (formatter = 'pxfwritable_export')
ENCODING = 6
DISTRIBUTED randomly;

INSERT INTO s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_october_september_ext
SELECT * FROM s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_october_september;

INSERT INTO s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_august_july_ext
SELECT * FROM s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_august_july;

INSERT INTO s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_june_may_ext
SELECT * FROM s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_june_may;

drop EXTERNAL table s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_october_september_ext;
drop EXTERNAL table s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_august_july_ext;
drop EXTERNAL table s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_june_may_ext;
'''
cur.execute(to_pxf)
conn.commit()

# Сбор Таргета Train

In [4]:
vsp_june_may = spark.read.parquet('hdfs://arnsdpsbx/user/team/team_ss/sales_vsp_mvs_june_may')
vsp_june_may.createOrReplaceTempView('sales_vsp_mvs_june_may')

In [5]:
vsp_august_july = spark.read.parquet('hdfs://arnsdpsbx/user/team/team_ss/sales_vsp_mvs_august_july')
vsp_august_july.createOrReplaceTempView('sales_vsp_mvs_august_july')

In [6]:
vsp_october_september = spark.read.parquet('hdfs://arnsdpsbx/user/team/team_ss/sales_vsp_mvs_october_september')
vsp_october_september.createOrReplaceTempView('sales_vsp_mvs_october_september')

In [7]:
sbol_june_may = spark.read.parquet('hdfs://arnsdpsbx/user/team/team_ss/sales_sbol_mvs_june_may')
sbol_june_may.createOrReplaceTempView('sales_sbol_mvs_june_may')

In [8]:
sbol_august_july = spark.read.parquet('hdfs://arnsdpsbx/user/team/team_ss/sales_sbol_mvs_august_july')
sbol_august_july.createOrReplaceTempView('sales_sbol_mvs_august_july')

In [9]:
sbol_october_september = spark.read.parquet('hdfs://arnsdpsbx/user/team/team_ss/sales_sbol_mvs_october_september')
sbol_october_september.createOrReplaceTempView('sales_sbol_mvs_october_september')

In [10]:
calls_october_september = spark.read.parquet('hdfs://arnsdpsbx/user/team/team_ss/calls_mvs_october_september')
calls_october_september.createOrReplaceTempView('calls_mvs_october_september')

In [11]:
calls_august_july = spark.read.parquet('hdfs://arnsdpsbx/user/team/team_ss/calls_mvs_august_july')
calls_august_july.createOrReplaceTempView('calls_mvs_august_july')

In [12]:
calls_june_may = spark.read.parquet('hdfs://arnsdpsbx/user/team/team_ss/calls_mvs_june_may')
calls_june_may.createOrReplaceTempView('calls_mvs_june_may')

In [25]:
km = pd.read_parquet('km_october_september.parquet')
km = km.drop_duplicates()
km = km.sample(9000000)
km_df = spark.createDataFrame(km)
km_df.createOrReplaceTempView('km_october_september')

/usr/sdp/current/spark3-client/python/pyspark/sql/pandas/conversion.py:371: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():


In [26]:
km = pd.read_parquet('km_august_july.parquet')
km = km.drop_duplicates()
km = km.sample(9000000)
km_df = spark.createDataFrame(km)
km_df.createOrReplaceTempView('km_august_july')

/usr/sdp/current/spark3-client/python/pyspark/sql/pandas/conversion.py:371: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():


In [27]:
km = pd.read_parquet('km_june_may.parquet')
km = km.drop_duplicates()
km = km.sample(9000000)
km_df = spark.createDataFrame(km)
km_df.createOrReplaceTempView('km_june_may')

/usr/sdp/current/spark3-client/python/pyspark/sql/pandas/conversion.py:371: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():


In [16]:
dkm = pd.read_parquet('dkm_october_september.parquet')
dkm = dkm.drop_duplicates()
dkm_df = spark.createDataFrame(dkm)
dkm_df.createOrReplaceTempView('dkm_october_september')

/usr/sdp/current/spark3-client/python/pyspark/sql/pandas/conversion.py:371: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():


In [17]:
dkm = pd.read_parquet('dkm_august_july.parquet')
dkm = dkm.drop_duplicates()
dkm_df = spark.createDataFrame(dkm)
dkm_df.createOrReplaceTempView('dkm_august_july')

/usr/sdp/current/spark3-client/python/pyspark/sql/pandas/conversion.py:371: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():


In [18]:
dkm = pd.read_parquet('dkm_june_may.parquet')
dkm = dkm.drop_duplicates()
dkm_df = spark.createDataFrame(dkm)
dkm_df.createOrReplaceTempView('dkm_june_may')

/usr/sdp/current/spark3-client/python/pyspark/sql/pandas/conversion.py:371: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():


In [28]:
telemarketing = pd.read_parquet('telemarketing_october_september.parquet')
telemarketing = telemarketing.drop_duplicates()
telemarketing = telemarketing.sample(7000000)
telemarketing_df = spark.createDataFrame(telemarketing)
telemarketing_df.createOrReplaceTempView('telemarketing_october_september')

/usr/sdp/current/spark3-client/python/pyspark/sql/pandas/conversion.py:371: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():


In [29]:
telemarketing = pd.read_parquet('telemarketing_august_july.parquet')
telemarketing = telemarketing.drop_duplicates()
telemarketing = telemarketing.sample(7000000)
telemarketing_df = spark.createDataFrame(telemarketing)
telemarketing_df.createOrReplaceTempView('telemarketing_august_july')

/usr/sdp/current/spark3-client/python/pyspark/sql/pandas/conversion.py:371: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():


In [21]:
telemarketing = pd.read_parquet('telemarketing_june_may.parquet')
telemarketing = telemarketing.drop_duplicates()
telemarketing = telemarketing.sample(7000000)
telemarketing_df = spark.createDataFrame(telemarketing)
telemarketing_df.createOrReplaceTempView('telemarketing_june_may')

/usr/sdp/current/spark3-client/python/pyspark/sql/pandas/conversion.py:371: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():


In [22]:
list_campain = ['ПУ Сберпремьер', 'Вклад', 'Перевод самозанятых в ИП',
       'Самозанятые', 'Накопительный счет', 'Потребительский кредит',
       'ПДС', 'Смена тарифа дебетовой карты',
       'Защита от клеща', 'SberPrime', 'Акцепт Спасибо',
       'Мобильная связь, Сбермобайл', 'Детская карта', 'SberPrimePlus',
       'Изменение лимита', 'Страхование. Защита квартиры', 'Каско',
       'Повышенный кэшбек Сберспасибо по ДК', 'Как зарплатный',
       'Сберпрайм Старт', 'Удержание от закрытия счета КК',
       'Защита дома Премьер. Квартира']

In [23]:
list_producttype = [None, 'Вклад', 'Кредитная карта', 'Накопительный счет',
       'SberPrime', 'ПДС', 'Потребительский кредит', 'Самозанятые',
       'ПУ Сберпремьер', 'Мобильная связь, Сбермобайл', 'Детская карта',
       'Акцепт Спасибо', 'SberPrimePlus', 'Смена тарифа дебетовой карты',
       'Защита от клеща', 'Как зарплатный', 'Перевод самозанятых в ИП',
       'Повышенный кэшбек Сберспасибо по ДК', 'Каско', 'Сберпрайм Старт']

In [30]:
spark.sql(f'''
with main_epk_october_september as (
    select epk_id, report_dt 
    from prx_bpm_client_aggr_custom_rozn_client_aggr.ft_client_aggr_mnth
    where report_dt = '2025-09-30' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS'    
),

dkm_october_september_sales as (
    select epk_id, sum(is_call_dkm) as sum_is_call_dkm, sum(is_call_s_dkm) as sum_is_call_s_dkm, sum(is_meet_dkm) as sum_is_meet_dkm
    from dkm_october_september
    group by epk_id 
),

km_october_september_sales as (
    select epk_id, sum(is_call_km) as sum_is_call_km, sum(is_call_s_km) as sum_is_call_s_km, sum(is_meet_km) as sum_is_meet_km
    from km_october_september
    group by epk_id
),

vsp_october_september_sales as (
    select epk_id, count(*) as count_sale_vsp 
    from sales_vsp_mvs_october_september 
    group by epk_id
),

sbol_october_september_sales as (
    select epk_id, count(*) as count_sale_sbol
    from sales_sbol_mvs_october_september 
    group by epk_id
),

calls_october_september_sales as (
    select epk_id, count(*) as count_sale_erkc
    from calls_mvs_october_september
    where channel = 'ЕРКЦ'
    group by epk_id
),

telemarketing_october_september_sales as (
    select clientid as epk_id, count(*) as count_sale_telemarketing
    from telemarketing_october_september
    where producttype in ({','.join([f"'{w}'" for w in list_producttype])}) 
    and campain in ({','.join([f"'{w}'" for w in list_campain])})
    and ppd_30_day_sale <= 10 and predlozhenie = 1 and ppp_30 = 1
    group by epk_id
),

final_october_september as (
    select * from main_epk_october_september
    left join dkm_october_september_sales using(epk_id)
    left join km_october_september_sales using(epk_id)
    left join vsp_october_september_sales using(epk_id)
    left join sbol_october_september_sales using(epk_id)
    left join telemarketing_october_september_sales using(epk_id)
    left join calls_october_september_sales using(epk_id)
),

oct_sep_target_going_self as (
    select epk_id, report_dt, '1' as target
    from final_october_september
    where count_sale_vsp >= 2 and (count_sale_sbol > 0 or count_sale_sbol is not null)
            and (count_sale_telemarketing = 0 or count_sale_telemarketing is null)
            and (count_sale_erkc = 0 or count_sale_erkc is null)
            and (sum_is_call_dkm = 0 or sum_is_call_dkm is null)
            and (sum_is_call_s_dkm = 0 or sum_is_call_s_dkm is null)
            and (sum_is_meet_dkm = 0 or sum_is_meet_dkm is null)
            and (sum_is_call_km = 0 or sum_is_call_km is null)
            and (sum_is_call_s_km = 0 or sum_is_call_s_km is null)
            and (sum_is_meet_km = 0 or sum_is_meet_km is null)
),

oct_sep_target_going_km as (
    select epk_id, report_dt, '2' as target
    from final_october_september
    where count_sale_vsp >= 2 and (count_sale_sbol = 0 or count_sale_sbol is null)
            and ((count_sale_telemarketing > 0 or count_sale_telemarketing is not null)
            or (count_sale_erkc > 0 or count_sale_erkc is not null)
            or (sum_is_call_dkm > 0 or sum_is_call_dkm is not null)
            or (sum_is_call_s_dkm > 0 or sum_is_call_s_dkm is not null)
            or (sum_is_meet_dkm > 0 or sum_is_meet_dkm is not null)
            or (sum_is_call_km > 0 or sum_is_call_km is not null)
            or (sum_is_call_s_km > 0 or sum_is_call_s_km is not null)
            or (sum_is_meet_km > 0 or sum_is_meet_km is not null))
),

oct_sep_target_not_going_self as (
    select epk_id, report_dt, '3' as target
    from final_october_september
    where count_sale_vsp < 2 and (count_sale_sbol > 0 or count_sale_sbol is not null)
            and (count_sale_telemarketing = 0 or count_sale_telemarketing is null)
            and (count_sale_erkc = 0 or count_sale_erkc is null)
            and (sum_is_call_dkm = 0 or sum_is_call_dkm is null)
            and (sum_is_call_s_dkm = 0 or sum_is_call_s_dkm is null)
            and (sum_is_meet_dkm = 0 or sum_is_meet_dkm is null)
            and (sum_is_call_km = 0 or sum_is_call_km is null)
            and (sum_is_call_s_km = 0 or sum_is_call_s_km is null)
            and (sum_is_meet_km = 0 or sum_is_meet_km is null)
),

oct_sep_target_not_going_km as (
    select epk_id, report_dt, '4' as target
    from final_october_september
    where count_sale_vsp < 2 and (count_sale_sbol = 0 or count_sale_sbol is null)
            and ((count_sale_telemarketing > 0 or count_sale_telemarketing is not null)
            or (count_sale_erkc > 0 or count_sale_erkc is not null)
            or (sum_is_call_dkm > 0 or sum_is_call_dkm is not null)
            or (sum_is_call_s_dkm > 0 or sum_is_call_s_dkm is not null)
            or (sum_is_meet_dkm > 0 or sum_is_meet_dkm is not null)
            or (sum_is_call_km > 0 or sum_is_call_km is not null)
            or (sum_is_call_s_km > 0 or sum_is_call_s_km is not null)
            or (sum_is_meet_km > 0 or sum_is_meet_km is not null))
),

oct_sep_target_nothing as (
    select epk_id, report_dt, '5' as target
    from final_october_september
    where (count_sale_vsp = 0 or count_sale_vsp is null)
            and (count_sale_sbol = 0 or count_sale_sbol is null)
            and (count_sale_telemarketing = 0 or count_sale_telemarketing is null)
            and (count_sale_erkc = 0 or count_sale_erkc is null)
            and (sum_is_call_dkm = 0 or sum_is_call_dkm is null)
            and (sum_is_call_s_dkm = 0 or sum_is_call_s_dkm is null)
            and (sum_is_meet_dkm = 0 or sum_is_meet_dkm is null)
            and (sum_is_call_km = 0 or sum_is_call_km is null)
            and (sum_is_call_s_km = 0 or sum_is_call_s_km is null)
            and (sum_is_meet_km = 0 or sum_is_meet_km is null)
)


(select * from oct_sep_target_going_self)
union all
(select * from oct_sep_target_going_km)
union all
(select * from oct_sep_target_not_going_self)
union all
(select * from oct_sep_target_not_going_km)
union all
(select * from oct_sep_target_nothing)
''').write.saveAsTable('arnsdpsbx_team_ss.bpm_premier_multiclass_new_iter_target_oct_sep', mode='overwrite')

In [31]:
spark.sql(f'''
with main_epk_august_july as (
    select epk_id, report_dt 
    from prx_bpm_client_aggr_custom_rozn_client_aggr.ft_client_aggr_mnth
    where report_dt = '2025-07-31' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS'    
),

dkm_august_july_sales as (
    select epk_id, sum(is_call_dkm) as sum_is_call_dkm, sum(is_call_s_dkm) as sum_is_call_s_dkm, sum(is_meet_dkm) as sum_is_meet_dkm
    from dkm_august_july 
    group by epk_id 
),

km_august_july_sales as (
    select epk_id, sum(is_call_km) as sum_is_call_km, sum(is_call_s_km) as sum_is_call_s_km, sum(is_meet_km) as sum_is_meet_km
    from km_august_july 
    group by epk_id
),

vsp_august_july_sales as (
    select epk_id, count(*) as count_sale_vsp 
    from sales_vsp_mvs_august_july 
    group by epk_id
),

sbol_august_july_sales as (
    select epk_id, count(*) as count_sale_sbol
    from sales_sbol_mvs_august_july
    group by epk_id
),

calls_august_july_sales as (
    select epk_id, count(*) as count_sale_erkc
    from calls_mvs_august_july 
    where channel = 'ЕРКЦ'
    group by epk_id
),

telemarketing_august_july_sales as (
    select clientid as epk_id, count(*) as count_sale_telemarketing
    from telemarketing_august_july
    where producttype in ({','.join([f"'{w}'" for w in list_producttype])}) 
    and campain in ({','.join([f"'{w}'" for w in list_campain])})
    and ppd_30_day_sale <= 10 and predlozhenie = 1 and ppp_30 = 1
    group by epk_id
),

final_august_july as (
    select * from main_epk_august_july
    left join dkm_august_july_sales using(epk_id)
    left join km_august_july_sales using(epk_id)
    left join vsp_august_july_sales using(epk_id)
    left join sbol_august_july_sales using(epk_id)
    left join telemarketing_august_july_sales using(epk_id)
    left join calls_august_july_sales using(epk_id)
),

aug_july_target_going_self as (
    select epk_id, report_dt, '1' as target
    from final_august_july
    where count_sale_vsp >= 2 and (count_sale_sbol > 0 or count_sale_sbol is not null)
            and (count_sale_telemarketing = 0 or count_sale_telemarketing is null)
            and (count_sale_erkc = 0 or count_sale_erkc is null)
            and (sum_is_call_dkm = 0 or sum_is_call_dkm is null)
            and (sum_is_call_s_dkm = 0 or sum_is_call_s_dkm is null)
            and (sum_is_meet_dkm = 0 or sum_is_meet_dkm is null)
            and (sum_is_call_km = 0 or sum_is_call_km is null)
            and (sum_is_call_s_km = 0 or sum_is_call_s_km is null)
            and (sum_is_meet_km = 0 or sum_is_meet_km is null)
),

aug_july_target_going_km as (
    select epk_id, report_dt, '2' as target
    from final_august_july
    where count_sale_vsp >= 2 and (count_sale_sbol = 0 or count_sale_sbol is null)
            and ((count_sale_telemarketing > 0 or count_sale_telemarketing is not null)
            or (count_sale_erkc > 0 or count_sale_erkc is not null)
            or (sum_is_call_dkm > 0 or sum_is_call_dkm is not null)
            or (sum_is_call_s_dkm > 0 or sum_is_call_s_dkm is not null)
            or (sum_is_meet_dkm > 0 or sum_is_meet_dkm is not null)
            or (sum_is_call_km > 0 or sum_is_call_km is not null)
            or (sum_is_call_s_km > 0 or sum_is_call_s_km is not null)
            or (sum_is_meet_km > 0 or sum_is_meet_km is not null))
),

aug_july_target_not_going_self as (
    select epk_id, report_dt, '3' as target
    from final_august_july
    where count_sale_vsp < 2 and (count_sale_sbol > 0 or count_sale_sbol is not null)
            and (count_sale_telemarketing = 0 or count_sale_telemarketing is null)
            and (count_sale_erkc = 0 or count_sale_erkc is null)
            and (sum_is_call_dkm = 0 or sum_is_call_dkm is null)
            and (sum_is_call_s_dkm = 0 or sum_is_call_s_dkm is null)
            and (sum_is_meet_dkm = 0 or sum_is_meet_dkm is null)
            and (sum_is_call_km = 0 or sum_is_call_km is null)
            and (sum_is_call_s_km = 0 or sum_is_call_s_km is null)
            and (sum_is_meet_km = 0 or sum_is_meet_km is null)
),

aug_july_target_not_going_km as (
    select epk_id, report_dt, '4' as target
    from final_august_july
    where count_sale_vsp < 2 and (count_sale_sbol = 0 or count_sale_sbol is null)
            and ((count_sale_telemarketing > 0 or count_sale_telemarketing is not null)
            or (count_sale_erkc > 0 or count_sale_erkc is not null)
            or (sum_is_call_dkm > 0 or sum_is_call_dkm is not null)
            or (sum_is_call_s_dkm > 0 or sum_is_call_s_dkm is not null)
            or (sum_is_meet_dkm > 0 or sum_is_meet_dkm is not null)
            or (sum_is_call_km > 0 or sum_is_call_km is not null)
            or (sum_is_call_s_km > 0 or sum_is_call_s_km is not null)
            or (sum_is_meet_km > 0 or sum_is_meet_km is not null))
),

aug_july_target_nothing as (
    select epk_id, report_dt, '5' as target
    from final_august_july
    where (count_sale_vsp = 0 or count_sale_vsp is null)
            and (count_sale_sbol = 0 or count_sale_sbol is null)
            and (count_sale_telemarketing = 0 or count_sale_telemarketing is null)
            and (count_sale_erkc = 0 or count_sale_erkc is null)
            and (sum_is_call_dkm = 0 or sum_is_call_dkm is null)
            and (sum_is_call_s_dkm = 0 or sum_is_call_s_dkm is null)
            and (sum_is_meet_dkm = 0 or sum_is_meet_dkm is null)
            and (sum_is_call_km = 0 or sum_is_call_km is null)
            and (sum_is_call_s_km = 0 or sum_is_call_s_km is null)
            and (sum_is_meet_km = 0 or sum_is_meet_km is null)
)



(select * from aug_july_target_going_self)
union all
(select * from aug_july_target_going_km)
union all
(select * from aug_july_target_not_going_self)
union all
(select * from aug_july_target_not_going_km)
union all
(select * from aug_july_target_nothing)

''').write.saveAsTable('arnsdpsbx_team_ss.bpm_premier_multiclass_new_iter_target_aug_july', mode='overwrite')

In [32]:
spark.sql(f'''
with main_epk_june_may as (
    select epk_id, report_dt 
    from prx_bpm_client_aggr_custom_rozn_client_aggr.ft_client_aggr_mnth
    where report_dt = '2025-05-31' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS'    
),

dkm_june_may_sales as (
    select epk_id, sum(is_call_dkm) as sum_is_call_dkm, sum(is_call_s_dkm) as sum_is_call_s_dkm, sum(is_meet_dkm) as sum_is_meet_dkm
    from dkm_june_may
    group by epk_id 
),

km_june_may_sales as (
    select epk_id, sum(is_call_km) as sum_is_call_km, sum(is_call_s_km) as sum_is_call_s_km, sum(is_meet_km) as sum_is_meet_km
    from km_june_may 
    group by epk_id
),

vsp_june_may_sales as (
    select epk_id, count(*) as count_sale_vsp 
    from sales_vsp_mvs_june_may
    group by epk_id
),

sbol_june_may_sales as (
    select epk_id, count(*) as count_sale_sbol
    from sales_sbol_mvs_june_may
    group by epk_id
),

calls_june_may_sales as (
    select epk_id, count(*) as count_sale_erkc
    from calls_mvs_june_may
    where channel = 'ЕРКЦ'
    group by epk_id
),

telemarketing_june_may_sales as (
    select clientid as epk_id, count(*) as count_sale_telemarketing
    from telemarketing_june_may
    where producttype in ({','.join([f"'{w}'" for w in list_producttype])}) 
    and campain in ({','.join([f"'{w}'" for w in list_campain])})
    and ppd_30_day_sale <= 10 and predlozhenie = 1 and ppp_30 = 1
    group by epk_id
),

final_june_may as (
    select * from main_epk_june_may
    left join dkm_june_may_sales using(epk_id)
    left join km_june_may_sales using(epk_id)
    left join vsp_june_may_sales using(epk_id)
    left join sbol_june_may_sales using(epk_id)
    left join telemarketing_june_may_sales using(epk_id)
    left join calls_june_may_sales using(epk_id)
),

june_may_target_going_self as (
    select epk_id, report_dt, '1' as target
    from final_june_may
    where count_sale_vsp >= 2 and (count_sale_sbol > 0 or count_sale_sbol is not null)
            and (count_sale_telemarketing = 0 or count_sale_telemarketing is null)
            and (count_sale_erkc = 0 or count_sale_erkc is null)
            and (sum_is_call_dkm = 0 or sum_is_call_dkm is null)
            and (sum_is_call_s_dkm = 0 or sum_is_call_s_dkm is null)
            and (sum_is_meet_dkm = 0 or sum_is_meet_dkm is null)
            and (sum_is_call_km = 0 or sum_is_call_km is null)
            and (sum_is_call_s_km = 0 or sum_is_call_s_km is null)
            and (sum_is_meet_km = 0 or sum_is_meet_km is null)
),

june_may_target_going_km as (
    select epk_id, report_dt, '2' as target
    from final_june_may
    where count_sale_vsp >= 2 and (count_sale_sbol = 0 or count_sale_sbol is null)
            and ((count_sale_telemarketing > 0 or count_sale_telemarketing is not null)
            or (count_sale_erkc > 0 or count_sale_erkc is not null)
            or (sum_is_call_dkm > 0 or sum_is_call_dkm is not null)
            or (sum_is_call_s_dkm > 0 or sum_is_call_s_dkm is not null)
            or (sum_is_meet_dkm > 0 or sum_is_meet_dkm is not null)
            or (sum_is_call_km > 0 or sum_is_call_km is not null)
            or (sum_is_call_s_km > 0 or sum_is_call_s_km is not null)
            or (sum_is_meet_km > 0 or sum_is_meet_km is not null))
),

june_may_target_not_going_self as (
    select epk_id, report_dt, '3' as target
    from final_june_may
    where count_sale_vsp < 2 and (count_sale_sbol > 0 or count_sale_sbol is not null)
            and (count_sale_telemarketing = 0 or count_sale_telemarketing is null)
            and (count_sale_erkc = 0 or count_sale_erkc is null)
            and (sum_is_call_dkm = 0 or sum_is_call_dkm is null)
            and (sum_is_call_s_dkm = 0 or sum_is_call_s_dkm is null)
            and (sum_is_meet_dkm = 0 or sum_is_meet_dkm is null)
            and (sum_is_call_km = 0 or sum_is_call_km is null)
            and (sum_is_call_s_km = 0 or sum_is_call_s_km is null)
            and (sum_is_meet_km = 0 or sum_is_meet_km is null)
),

june_may_target_not_going_km as (
    select epk_id, report_dt, '4' as target
    from final_june_may
    where count_sale_vsp < 2 and (count_sale_sbol = 0 or count_sale_sbol is null)
            and ((count_sale_telemarketing > 0 or count_sale_telemarketing is not null)
            or (count_sale_erkc > 0 or count_sale_erkc is not null)
            or (sum_is_call_dkm > 0 or sum_is_call_dkm is not null)
            or (sum_is_call_s_dkm > 0 or sum_is_call_s_dkm is not null)
            or (sum_is_meet_dkm > 0 or sum_is_meet_dkm is not null)
            or (sum_is_call_km > 0 or sum_is_call_km is not null)
            or (sum_is_call_s_km > 0 or sum_is_call_s_km is not null)
            or (sum_is_meet_km > 0 or sum_is_meet_km is not null))
),

june_may_target_nothing as (
    select epk_id, report_dt, '5' as target
    from final_june_may
    where (count_sale_vsp = 0 or count_sale_vsp is null)
            and (count_sale_sbol = 0 or count_sale_sbol is null)
            and (count_sale_telemarketing = 0 or count_sale_telemarketing is null)
            and (count_sale_erkc = 0 or count_sale_erkc is null)
            and (sum_is_call_dkm = 0 or sum_is_call_dkm is null)
            and (sum_is_call_s_dkm = 0 or sum_is_call_s_dkm is null)
            and (sum_is_meet_dkm = 0 or sum_is_meet_dkm is null)
            and (sum_is_call_km = 0 or sum_is_call_km is null)
            and (sum_is_call_s_km = 0 or sum_is_call_s_km is null)
            and (sum_is_meet_km = 0 or sum_is_meet_km is null)
)


(select * from june_may_target_going_self)
union all
(select * from june_may_target_going_km)
union all
(select * from june_may_target_not_going_self)
union all
(select * from june_may_target_not_going_km)
union all
(select * from june_may_target_nothing)
''').write.saveAsTable('arnsdpsbx_team_ss.bpm_premier_multiclass_new_iter_target_june_may', mode='overwrite')

In [33]:
spark.sql('''
with answer as (
(select * from arnsdpsbx_team_ss.bpm_premier_multiclass_new_iter_target_oct_sep)
union all 
(select * from arnsdpsbx_team_ss.bpm_premier_multiclass_new_iter_target_aug_july)
union all
(select * from arnsdpsbx_team_ss.bpm_premier_multiclass_new_iter_target_june_may)
)

select * from answer 
where answer.epk_id in (
        select epk_id from answer 
        group by epk_id
        having count(*) = 1
    )
order by random()
limit 800000
''').write.saveAsTable('arnsdpsbx_team_ss.bpm_premier_multiclass_new_target_v3_new', mode='overwrite')

In [34]:
spark.sql('''
select report_dt, target, count(epk_id), count(distinct epk_id) from arnsdpsbx_team_ss.bpm_premier_multiclass_new_target_v3_new
group by report_dt, target
order by report_dt, target
''').show()

+----------+------+-------------+----------------------+
| report_dt|target|count(epk_id)|count(DISTINCT epk_id)|
+----------+------+-------------+----------------------+
|2025-05-31|     1|        25710|                 25710|
|2025-05-31|     2|        45949|                 45949|
|2025-05-31|     3|        15523|                 15523|
|2025-05-31|     4|        12909|                 12909|
|2025-05-31|     5|       186441|                186441|
|2025-07-31|     1|        12135|                 12135|
|2025-07-31|     2|        19515|                 19515|
|2025-07-31|     3|         9199|                  9199|
|2025-07-31|     4|         9917|                  9917|
|2025-07-31|     5|       202987|                202987|
|2025-09-30|     1|        41759|                 41759|
|2025-09-30|     2|        91207|                 91207|
|2025-09-30|     3|        14675|                 14675|
|2025-09-30|     4|        25523|                 25523|
|2025-09-30|     5|        8655

In [35]:
spark.sql('''
select count(epk_id), count(distinct epk_id) from arnsdpsbx_team_ss.bpm_premier_multiclass_new_target_v3_new
''').show()

+-------------+----------------------+
|count(epk_id)|count(DISTINCT epk_id)|
+-------------+----------------------+
|       800000|                800000|
+-------------+----------------------+



# Сбор Таргета OOT

## Telemarketing

In [4]:
df_december_november = pd.read_sql('''
with first as (select epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-11-30' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select clientid, predlozhenie, ppp_30, ppd_30_day_sale, splits, producttype, campain, x_btn_connect_date 
from prx_bpm_telemarketing_analitika_s_grnplm_as_rozn_anofl_view_3t2."v_anofl_rep$_coldfun_sale_deteil_dp1_21417984_view" a 
inner join first b on a.clientid = b.epk_id
where x_btn_connect_date >= '2024-12-01' and x_btn_connect_date <= '2025-11-30'
and splits not in ('SC_SaleKr','SaleKR_TM','Hot_sales', 'Sale_TM')   -- условие которым исключаем сплиты горячего потока
and coalesce (producttype,'') <> 'Изменение лимита'  --исключаем кампанию, тк кампания сервисная
and coalesce (producttype,'') <> 'Мегамаркет'   --исключаем кампанию, тк кампания сервисная
and campain is not null   -- учитываются только клиенты с разметкой в кампании, расчет всех метрик  учетом этого условия
;
''', conn)

/tmp/ipykernel_1789158/3530644794.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_december_november = pd.read_sql('''


In [5]:
df_december_november = df_december_november.drop_duplicates()

In [6]:
df_december_november.to_parquet('telemarketing_december_november.parquet')

## Звонки

In [7]:
q = f'''
DROP TABLE IF EXISTS s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_december_november;
    
CREATE TABLE s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_december_november as (
WITH base as (
    select epk_id, main_call, channel, presentation_flag, contact_flag, sale_dt_1prod, sale_dt_2prod
from prx_bpm_zvonki_i_prodazhi_gp_s_grnplm_as_rozn_anofl_view_20t."v_anofl_rep$_tm_funnel_dp1_21417984_view"
where main_call between '2024-12-01' and '2025-11-30'
)
SELECT
*
FROM base
)
'''
conn.rollback()
cur.execute(q)
conn.commit()

In [4]:
to_pxf = f'''
CREATE WRITABLE EXTERNAL TABLE s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_december_november_ext
(
   LIKE s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_december_november
)
LOCATION ('pxf://user/team/team_ss/calls_mvs_december_novemberr?PROFILE=hdfs:parquet&SERVER=sbx_sdp_ld_rozn_electron')
ON ALL
FORMAT 'custom' (formatter = 'pxfwritable_export')
ENCODING = 6
DISTRIBUTED randomly;


INSERT INTO s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_december_november_ext
SELECT * FROM s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_december_november;


drop EXTERNAL table s_grnplm_ld_rozn_electron_ss_users_temp.calls_mvs_december_november_ext;
'''
cur.execute(to_pxf)
conn.commit()

## KM

In [8]:
df_december_november = pd.read_sql('''
select 
epk_id, activity_dt,
sum(case when activity_type = 'Исходящий звонок' and activity_status_detail in ('Звонок состоялся', 'Звонок отработан') then 1 else 0 end) as is_call_km,  -- звонков
sum(case when activity_type = 'Исходящий звонок' and activity_status_detail in ('Звонок состоялся', 'Звонок отработан') and date_part('epoch', factend_dttm - factstart_dttm) >= 60 then 1 else 0 end) as is_call_s_km,  --дозвоны
sum(case when activity_status_detail = 'Встреча состоялась' and date_part('epoch', factend_dttm - factstart_dttm) >= 60 then 1 else 0 end) as is_meet_km --встречи
from s_grnplm_ld_rozn_electron_mvs.prmr_data_pprb_activity a_p
where 1=1 
and activity_dt between '2024-12-01' and '2025-11-30'
and a_p.role_from_sap = 'ПРЕМЬЕР' and a_p.activity_status in ('Выполнена', 'Завершена', 'Закрыта')
and a_p.activity_type in ('Исходящий звонок','Внутренняя встреча')
group by 1,2

''', conn)

/tmp/ipykernel_1789158/2065355526.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_december_november = pd.read_sql('''


In [9]:
df_december_november = df_december_november.drop_duplicates()

In [10]:
df_december_november.to_parquet('km_december_november.parquet')

## ДКМ

In [11]:
df_december_november = pd.read_sql('''
with first as (select distinct epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-11-30' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select a.clientid as epk_id, --формат text
"Дата" as activity_dt,
sum(case when a."время звонка" > 0 then 1 else 0 end) as is_call_dkm,
sum(case when a."время звонка" >= 1 or (a."время звонка" > 0 and a."Статус звонка" = 'Дозвон') then 1 else 0 end) as is_call_s_dkm,
0 as is_meet_dkm  --нет встреч
from s_grnplm_ld_rozn_electron_mvs.v_dkm_calls a
where a."Дата" between '2024-12-01' and '2025-11-30'
group by 1,2
''', conn)

/tmp/ipykernel_1789158/602820053.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_december_november = pd.read_sql('''


In [12]:
df_december_november = df_december_november.drop_duplicates()

In [13]:
df_december_november.to_parquet('dkm_december_november.parquet')

## Sales MVS SBOL and VSP

### SBOL

In [14]:
q = f'''
DROP TABLE IF EXISTS s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_december_november;
    
CREATE TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_december_november as (
with first as (select distinct epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-11-30' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select epk_id, sale_dt, product_class_name, sale_channel_name from s_grnplm_vd_rozn_mpp_aaas_vd.ft_prod_sales
inner join first using(epk_id)
where sale_dt between '2024-12-01' and '2025-11-30'
and product_class_name in (
    'Запрос кредитной истории',
    'СберСпасибо',
    'Текущий счет',
    'Зарплатный клиент',
    'Монеты и слитки',
    'Страховой продукт',
    'Брокерское обслуживание (CIB)',
    'Платежи',
    'Срочный депозит',
    'Подписка',
    'Металлический счет',
    'Маркировка в канале Премьер/VIP',
    'Карточный счет',
    'Потребительский кредит',
    'Пакет услуг Премьер',
    'ПИФ/ДУ',
    'АП ЖКХ (новая система)',
    'Переводы',
    'Открытие счета ЮЛ',
    'Подключение транзакционных пушей',
    'Облигация',
    'Кредитная карта',
    'Автокредит',
    'Пенсионные начисления'
    )
and sale_channel_name in ( 
    'Интернет-банк (СБОЛ)',
    'СБОЛ.Про',
    'Сайт',
    'СБОЛ',
    'Интернет сайт',
    'MOBILE СБОЛ',
    'SBOL PRO',
    'SBOL Mob',
    'SBOL Web',
    'САЙТ',
    'СБОЛ МП',
    'СБЕРБАНК ПРЕМЬЕР-SBOL PRO',
    'Мобильное приложение СБОЛ',
    'СБОЛ САЙТ',
    'WEB СБОЛ',
    'Сайт Банка(Гостевой СБОЛ)',
    'Мобильный банк'
    )
)
'''
conn.rollback()
cur.execute(q)
conn.commit()

In [15]:
to_pxf = f'''


CREATE WRITABLE EXTERNAL TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_december_november_ext
(
   LIKE s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_december_november
)
LOCATION ('pxf://user/team/team_ss/sales_sbol_mvs_december_november?PROFILE=hdfs:parquet&SERVER=sbx_sdp_ld_rozn_electron')
ON ALL
FORMAT 'custom' (formatter = 'pxfwritable_export')
ENCODING = 6
DISTRIBUTED randomly;


INSERT INTO s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_december_november_ext
SELECT * FROM s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_december_november;


drop EXTERNAL table s_grnplm_ld_rozn_electron_ss_users_temp.sales_sbol_mvs_december_november_ext;
'''
cur.execute(to_pxf)
conn.commit()

###  VSP

In [16]:
q = f'''
DROP TABLE IF EXISTS s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_december_november;
    
CREATE TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_december_november as (
with first as (select distinct epk_id from s_grnplm_vd_rozn_mpp_daas_hdp_vd.ft_client_aggr_mnth
where report_dt = '2025-11-30' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS')

select epk_id, sale_dt, product_class_name, sale_channel_name from s_grnplm_vd_rozn_mpp_aaas_vd.ft_prod_sales
inner join first using(epk_id)
where sale_dt between '2024-12-01' and '2025-11-30'
and product_class_name in (
    'Запрос кредитной истории',
    'СберСпасибо',
    'Текущий счет',
    'Зарплатный клиент',
    'Монеты и слитки',
    'Страховой продукт',
    'Брокерское обслуживание (CIB)',
    'Платежи',
    'Срочный депозит',
    'Подписка',
    'Металлический счет',
    'Маркировка в канале Премьер/VIP',
    'Карточный счет',
    'Потребительский кредит',
    'Пакет услуг Премьер',
    'ПИФ/ДУ',
    'АП ЖКХ (новая система)',
    'Переводы',
    'Открытие счета ЮЛ',
    'Подключение транзакционных пушей',
    'Облигация',
    'Кредитная карта',
    'Автокредит',
    'Пенсионные начисления'
    )
and sale_channel_name in ( 
    'ПРЯМЫЕ ПРОДАЖИ',
    'СБ1',
    'Масс',
    'ЕФС ВИП обслуживание',
    'Прямые продажи на предприятии (DSA)',
    'Сайт',
    'SB1',
    'PRIVATEBANKING',
    'ПРЯМЫЕ ПРОДАЖИ ВИП',
    'Premier',
    'САЙТ',
    'Клиентская зона',
    'PB',
    'MASS',
    'ВСП',
    'VIP',
    'ВСП Премьер',
    'ВИП ВСП',
    'Премьер',
    'ВСП. Мотивация',
    'Офисы Private Banking',
    'СБЕРБАНК ПРЕМЬЕР-ПАО'
    )
order by random()
limit 30000000
)
'''
conn.rollback()
cur.execute(q)
conn.commit()

In [17]:
to_pxf = f'''


CREATE WRITABLE EXTERNAL TABLE s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_december_november_ext
(
   LIKE s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_december_november
)
LOCATION ('pxf://user/team/team_ss/sales_vsp_mvs_december_november?PROFILE=hdfs:parquet&SERVER=sbx_sdp_ld_rozn_electron')
ON ALL
FORMAT 'custom' (formatter = 'pxfwritable_export')
ENCODING = 6
DISTRIBUTED randomly;


INSERT INTO s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_december_november_ext
SELECT * FROM s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_december_november;

drop EXTERNAL table s_grnplm_ld_rozn_electron_ss_users_temp.sales_vsp_mvs_december_november_ext;
'''
cur.execute(to_pxf)
conn.commit()

## all

In [5]:
vsp_nov_oct = spark.read.parquet('hdfs://arnsdpsbx/user/team/team_ss/sales_vsp_mvs_december_november')
vsp_nov_oct.createOrReplaceTempView('sales_vsp_mvs_december_november')

In [6]:
sbol_nov_oct = spark.read.parquet('hdfs://arnsdpsbx/user/team/team_ss/sales_sbol_mvs_december_november')
sbol_nov_oct.createOrReplaceTempView('sales_sbol_mvs_december_november')

In [8]:
calls_nov_oct = spark.read.parquet('hdfs://arnsdpsbx/user/team/team_ss/calls_mvs_december_novemberr')
calls_nov_oct.createOrReplaceTempView('calls_mvs_december_november')

In [18]:
km = pd.read_parquet('km_december_november.parquet')
km = km.drop_duplicates()
km = km.sample(9000000)
km_df = spark.createDataFrame(km)
km_df.createOrReplaceTempView('km_december_november')

/usr/sdp/current/spark3-client/python/pyspark/sql/pandas/conversion.py:371: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():


In [10]:
dkm = pd.read_parquet('dkm_december_november.parquet')
dkm = dkm.drop_duplicates()
dkm_df = spark.createDataFrame(dkm)
dkm_df.createOrReplaceTempView('dkm_december_november')

/usr/sdp/current/spark3-client/python/pyspark/sql/pandas/conversion.py:371: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():


In [19]:
telemarketing = pd.read_parquet('telemarketing_december_november.parquet')
telemarketing = telemarketing.drop_duplicates()
telemarketing = telemarketing.sample(9000000)
telemarketing_df = spark.createDataFrame(telemarketing)
telemarketing_df.createOrReplaceTempView('telemarketing_december_november')

/usr/sdp/current/spark3-client/python/pyspark/sql/pandas/conversion.py:371: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():


In [12]:
list_campain = ['ПУ Сберпремьер', 'Вклад', 'Перевод самозанятых в ИП',
       'Самозанятые', 'Накопительный счет', 'Потребительский кредит',
       'ПДС', 'Смена тарифа дебетовой карты',
       'Защита от клеща', 'SberPrime', 'Акцепт Спасибо',
       'Мобильная связь, Сбермобайл', 'Детская карта', 'SberPrimePlus',
       'Изменение лимита', 'Страхование. Защита квартиры', 'Каско',
       'Повышенный кэшбек Сберспасибо по ДК', 'Как зарплатный',
       'Сберпрайм Старт', 'Удержание от закрытия счета КК',
       'Защита дома Премьер. Квартира']

In [13]:
list_producttype = [None, 'Вклад', 'Кредитная карта', 'Накопительный счет',
       'SberPrime', 'ПДС', 'Потребительский кредит', 'Самозанятые',
       'ПУ Сберпремьер', 'Мобильная связь, Сбермобайл', 'Детская карта',
       'Акцепт Спасибо', 'SberPrimePlus', 'Смена тарифа дебетовой карты',
       'Защита от клеща', 'Как зарплатный', 'Перевод самозанятых в ИП',
       'Повышенный кэшбек Сберспасибо по ДК', 'Каско', 'Сберпрайм Старт']

In [20]:
spark.sql(f'''
with main_epk_december_november as (
    select epk_id, report_dt 
    from prx_bpm_client_aggr_custom_rozn_client_aggr.ft_client_aggr_mnth
    where report_dt = '2025-11-30' and sd_dead_nflag = 0 and cla_full_active_nflag = 1 and seg_client_cx_segment_cd = 'MVS'    
),

dkm_december_november_sales as (
    select epk_id, sum(is_call_dkm) as sum_is_call_dkm, sum(is_call_s_dkm) as sum_is_call_s_dkm, sum(is_meet_dkm) as sum_is_meet_dkm
    from dkm_december_november
    group by epk_id 
),

km_december_november_sales as (
    select epk_id, sum(is_call_km) as sum_is_call_km, sum(is_call_s_km) as sum_is_call_s_km, sum(is_meet_km) as sum_is_meet_km
    from km_december_november
    group by epk_id
),

vsp_december_november_sales as (
    select epk_id, count(*) as count_sale_vsp 
    from sales_vsp_mvs_december_november
    group by epk_id
),

sbol_december_november_sales as (
    select epk_id, count(*) as count_sale_sbol
    from sales_sbol_mvs_december_november
    group by epk_id
),

calls_december_november_sales as (
    select epk_id, count(*) as count_sale_erkc
    from calls_mvs_december_november
    where channel = 'ЕРКЦ'
    group by epk_id
),

telemarketing_december_november_sales as (
    select clientid as epk_id, count(*) as count_sale_telemarketing
    from telemarketing_december_november
    where producttype in ({','.join([f"'{w}'" for w in list_producttype])}) 
    and campain in ({','.join([f"'{w}'" for w in list_campain])})
    and ppd_30_day_sale <= 10 and predlozhenie = 1 and ppp_30 = 1
    group by epk_id
),

final_december_november as (
    select * from main_epk_december_november
    left join dkm_december_november_sales using(epk_id)
    left join km_december_november_sales using(epk_id)
    left join vsp_december_november_sales using(epk_id)
    left join sbol_december_november_sales using(epk_id)
    left join telemarketing_december_november_sales using(epk_id)
    left join calls_december_november_sales using(epk_id)
),

oct_sep_target_going_self as (
    select epk_id, report_dt, '1' as target
    from final_december_november
    where count_sale_vsp >= 2 and (count_sale_sbol > 0 or count_sale_sbol is not null)
            and (count_sale_telemarketing = 0 or count_sale_telemarketing is null)
            and (count_sale_erkc = 0 or count_sale_erkc is null)
            and (sum_is_call_dkm = 0 or sum_is_call_dkm is null)
            and (sum_is_call_s_dkm = 0 or sum_is_call_s_dkm is null)
            and (sum_is_meet_dkm = 0 or sum_is_meet_dkm is null)
            and (sum_is_call_km = 0 or sum_is_call_km is null)
            and (sum_is_call_s_km = 0 or sum_is_call_s_km is null)
            and (sum_is_meet_km = 0 or sum_is_meet_km is null)
),

oct_sep_target_going_km as (
    select epk_id, report_dt, '2' as target
    from final_december_november
    where count_sale_vsp >= 2 and (count_sale_sbol = 0 or count_sale_sbol is null)
            and ((count_sale_telemarketing > 0 or count_sale_telemarketing is not null)
            or (count_sale_erkc > 0 or count_sale_erkc is not null)
            or (sum_is_call_dkm > 0 or sum_is_call_dkm is not null)
            or (sum_is_call_s_dkm > 0 or sum_is_call_s_dkm is not null)
            or (sum_is_meet_dkm > 0 or sum_is_meet_dkm is not null)
            or (sum_is_call_km > 0 or sum_is_call_km is not null)
            or (sum_is_call_s_km > 0 or sum_is_call_s_km is not null)
            or (sum_is_meet_km > 0 or sum_is_meet_km is not null))
),

oct_sep_target_not_going_self as (
    select epk_id, report_dt, '3' as target
    from final_december_november
    where count_sale_vsp < 2 and (count_sale_sbol > 0 or count_sale_sbol is not null)
            and (count_sale_telemarketing = 0 or count_sale_telemarketing is null)
            and (count_sale_erkc = 0 or count_sale_erkc is null)
            and (sum_is_call_dkm = 0 or sum_is_call_dkm is null)
            and (sum_is_call_s_dkm = 0 or sum_is_call_s_dkm is null)
            and (sum_is_meet_dkm = 0 or sum_is_meet_dkm is null)
            and (sum_is_call_km = 0 or sum_is_call_km is null)
            and (sum_is_call_s_km = 0 or sum_is_call_s_km is null)
            and (sum_is_meet_km = 0 or sum_is_meet_km is null)
),

oct_sep_target_not_going_km as (
    select epk_id, report_dt, '4' as target
    from final_december_november
    where count_sale_vsp < 2 and (count_sale_sbol = 0 or count_sale_sbol is null)
            and ((count_sale_telemarketing > 0 or count_sale_telemarketing is not null)
            or (count_sale_erkc > 0 or count_sale_erkc is not null)
            or (sum_is_call_dkm > 0 or sum_is_call_dkm is not null)
            or (sum_is_call_s_dkm > 0 or sum_is_call_s_dkm is not null)
            or (sum_is_meet_dkm > 0 or sum_is_meet_dkm is not null)
            or (sum_is_call_km > 0 or sum_is_call_km is not null)
            or (sum_is_call_s_km > 0 or sum_is_call_s_km is not null)
            or (sum_is_meet_km > 0 or sum_is_meet_km is not null))
),

oct_sep_target_nothing as (
    select epk_id, report_dt, '5' as target
    from final_december_november
    where (count_sale_vsp = 0 or count_sale_vsp is null)
            and (count_sale_sbol = 0 or count_sale_sbol is null)
            and (count_sale_telemarketing = 0 or count_sale_telemarketing is null)
            and (count_sale_erkc = 0 or count_sale_erkc is null)
            and (sum_is_call_dkm = 0 or sum_is_call_dkm is null)
            and (sum_is_call_s_dkm = 0 or sum_is_call_s_dkm is null)
            and (sum_is_meet_dkm = 0 or sum_is_meet_dkm is null)
            and (sum_is_call_km = 0 or sum_is_call_km is null)
            and (sum_is_call_s_km = 0 or sum_is_call_s_km is null)
            and (sum_is_meet_km = 0 or sum_is_meet_km is null)
)


(select * from oct_sep_target_going_self)
union all
(select * from oct_sep_target_going_km)
union all
(select * from oct_sep_target_not_going_self)
union all
(select * from oct_sep_target_not_going_km)
union all
(select * from oct_sep_target_nothing)
''').write.saveAsTable('arnsdpsbx_team_ss.bpm_premier_multiclass_new_target_v3_oot', mode='overwrite')

In [21]:
spark.sql('''
select report_dt, target, count(epk_id), count(distinct epk_id) from arnsdpsbx_team_ss.bpm_premier_multiclass_new_target_v3_oot
group by report_dt, target
order by report_dt, target
''').show()

+----------+------+-------------+----------------------+
| report_dt|target|count(epk_id)|count(DISTINCT epk_id)|
+----------+------+-------------+----------------------+
|2025-11-30|     1|       211664|                211664|
|2025-11-30|     2|       577935|                577935|
|2025-11-30|     3|        95559|                 95559|
|2025-11-30|     4|        93884|                 93884|
|2025-11-30|     5|       607657|                607657|
+----------+------+-------------+----------------------+

